# SOC-Triage-Gym v2 — GRPO Training

**Hackathon: OpenEnv Apr 2026 — Theme #1 Multi-Agent Interactions + Fleet AI Oversight**

This notebook trains a Tier-1 SOC Analyst LLM on multi-agent team episodes using **GRPO** (Group Relative Policy Optimization).

### Environment Architecture
```
Alert Queue → [Tier-1 Analyst] → ESCALATE_TO_TIER2 → [Tier-2 Responder] → [SOC Manager]
               (40 steps)           (ticket bus)        (20 steps)          (8 steps)
                                                                    ↑
                                                         [Red-Team Generator]
                                                         (adaptive curriculum)
```

### Reward Structure
- **Per step**: `0.6 × role_specific + 0.4 × team_F1`
- **Episode final**: `0.6 × individual + 0.4 × team_F1` (prevents GRPO advantage collapse)

### Training Plan
| Phase | Role | Frozen Roles | GPU Time |
|-------|------|-------------|----------|
| Day 1 AM | Tier-1 | Scripted T2+Manager | ~4h H100 |
| Day 1 PM | Tier-2 | Trained T1 | ~3h H100 |
| Day 2 AM | Manager | Trained T1+T2 | ~2h H100 |
| Day 2 PM | Joint | All co-trained | ~4h H100 |

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


In [ ]:
# Mount Google Drive so trained weights and plots persist across sessions.
from google.colab import drive
drive.mount('/content/drive')


## 1. Install Dependencies

> **Before running cell 1:** Runtime → Change runtime type → **GPU** (T4 or better)
>
> **After running cells 1 + 2:** Runtime → **Restart session** → then continue from cell 3.
>
> unsloth patches the Python environment at install time and must be restarted before it can be imported.

In [ ]:
# ── Step 1: unsloth (Colab-compatible install from source) ──────────────────
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# ── Step 2: TRL + training stack ─────────────────────────────────────────────
# NOTES:
# - No mergekit / llm_blender / weave: we stub them in cell 3 (pydantic pin conflict).
# - pydantic>=2.11: TRL 0.24 uses pydantic models with torch.Tensor fields.
# - transformers<5: transformers 5.0 removed compat attrs (warnings_issued) that
#   PEFT/TRL still rely on. Pin until the stack catches up.
!pip install trl peft accelerate datasets "pydantic>=2.11" "transformers<5" -q

# ── Step 3: environment deps ─────────────────────────────────────────────────
!pip install fastapi uvicorn httpx matplotlib numpy -q

# ── Step 4: clone repo if not already present ───────────────────────────────
import os
if not os.path.exists('server') and not os.path.exists('openenv-2'):
    !git clone https://github.com/ROHITCRAFTSYT/-Metas-OpenEnv-2.git openenv-2 -q
    os.chdir('openenv-2')
elif os.path.exists('openenv-2') and not os.path.exists('server'):
    os.chdir('openenv-2')

print('✓ Install complete. Working dir:', os.getcwd())


In [ ]:
# ── Verify installs (unsloth is checked later — it needs a runtime restart) ──
import importlib, sys, os

# After a Runtime Restart, Colab resets CWD to /content. Re-enter the repo
# so later cells can find server.app, train_grpo, scenarios, etc.
if os.path.exists('openenv-2') and not os.path.exists('server'):
    os.chdir('openenv-2')
print('Working dir:', os.getcwd())

# ── Install stubs for optional TRL deps ──────────────────────────────────────
# TRL 0.24+ does lazy `from X... import ...` inside grpo_trainer for features
# we don't use (model-merging, LLM-as-judge blender, W&B Weave telemetry).
# Installing them pins pydantic or pulls enormous deps. This meta_path finder
# returns a MagicMock for any import starting with a stubbed prefix — TRL's
# import succeeds, we never touch the functionality.
import importlib.abc, importlib.machinery
from unittest.mock import MagicMock

_STUBBED_MODULES = ('mergekit', 'llm_blender', 'weave')

class _OptionalTRLStubLoader(importlib.abc.Loader):
    def create_module(self, spec):
        m = MagicMock()
        m.__name__ = spec.name
        m.__path__ = []
        m.__spec__ = spec
        return m
    def exec_module(self, module):
        pass

class _OptionalTRLStubFinder(importlib.abc.MetaPathFinder):
    def find_spec(self, fullname, path, target=None):
        for stub in _STUBBED_MODULES:
            if fullname == stub or fullname.startswith(stub + '.'):
                return importlib.machinery.ModuleSpec(fullname, _OptionalTRLStubLoader())
        return None

if not any(isinstance(f, _OptionalTRLStubFinder) for f in sys.meta_path):
    sys.meta_path.insert(0, _OptionalTRLStubFinder())
    print(f'✓ Stubs installed for: {", ".join(_STUBBED_MODULES)}')

checks = {
    'trl': 'trl',
    'peft': 'peft',
    'transformers': 'transformers',
    'fastapi': 'fastapi',
    'httpx': 'httpx',
    'pydantic': 'pydantic',
}
all_ok = True
for label, pkg in checks.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', getattr(mod, 'VERSION', 'unknown'))
        print(f'  ✓ {label} {ver}')
    except ImportError:
        print(f'  ✗ {label} MISSING — rerun the install cell', file=sys.stderr)
        all_ok = False

# Sanity check: pydantic must be >=2.11 for TRL 0.24's grpo_trainer.
import pydantic
_pv = tuple(int(x) for x in str(pydantic.VERSION).split('.')[:2])
if _pv < (2, 11):
    raise RuntimeError(
        f'pydantic {pydantic.VERSION} is too old for TRL 0.24+. '
        f'Run: !pip install -U "pydantic>=2.11" — then Runtime → Restart session.'
    )

# transformers 5.0 removed `warnings_issued` that PEFT/TRL depend on.
import transformers
_tv = tuple(int(x) for x in str(transformers.__version__).split('.')[:2])
if _tv >= (5, 0):
    print(f'  ⚠ transformers {transformers.__version__} may break PEFT/TRL. '
          f'Run: !pip install "transformers<5" -q and restart.')

if not all_ok:
    raise RuntimeError('Some packages are missing. Re-run cell 2 and check for errors.')

print('\n✓ Base packages present.')
print('\n⚠️  If you have NOT yet restarted after the install cell:')
print('    Runtime → Restart session, then re-run THIS cell and continue.')
print('    (unsloth patches the Python environment and only takes effect after restart)')


## 2. Start SOC-Triage-Gym Server

In [1]:
import subprocess, time, httpx, sys, os

if os.path.exists('openenv-2') and not os.path.exists('server'):
    os.chdir('openenv-2')
assert os.path.exists('server'), f'server/ not found in {os.getcwd()} — run the install cell first'

# Redirect server logs to a file instead of subprocess.PIPE.
# subprocess.PIPE has a ~64KB OS buffer that NOTHING is draining;
# once it fills, the server blocks on every log line and all /step
# calls hang until the client timeout fires.
_server_log = open('server.log', 'w')
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '7860',
     '--log-level', 'warning'],   # also: drop log volume so disk grows slowly
    stdout=_server_log, stderr=subprocess.STDOUT
)

client = httpx.Client(base_url='http://localhost:7860', timeout=120.0)

print('Waiting for server...', end='', flush=True)
for attempt in range(30):
    time.sleep(2)
    if server_proc.poll() is not None:
        _server_log.close()
        with open('server.log') as f:
            err = f.read()[-3000:]
        raise RuntimeError(f'Server process exited early:\n{err}')
    try:
        resp = client.get('/health')
        if resp.status_code == 200:
            print(f' ready after {(attempt+1)*2}s')
            print('Health:', resp.json())
            break
    except (httpx.ConnectError, httpx.ReadTimeout, httpx.ConnectTimeout):
        print('.', end='', flush=True)
else:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_proc.kill()
    _server_log.close()
    with open('server.log') as f:
        raise RuntimeError(f'Server never became healthy.\nstderr:\n{f.read()[-3000:]}')

Waiting for server... ready after 2s
Health: {'status': 'healthy', 'version': '0.1.0', 'env': 'soc-triage-gym'}


## 3. Verify Environment — Run Sample Episode

In [2]:
import json, httpx

# 120s timeout: if the LLM judge is enabled and a call is slow, this gives it
# headroom. The judge itself has a 5s internal timeout, but episodes with many
# manager steps can still exceed 30s cumulatively.
client = httpx.Client(base_url='http://localhost:7860', timeout=120.0)

# Test 1: Legacy solo mode (backward compat)
obs = client.post('/reset', json={'task_id': 'phishing', 'seed': 42}).json()
print(f"Solo mode — alerts: {len(obs['alert_queue'])}, mode: {obs.get('episode_mode')}")

# Test 2: Team mode
obs = client.post('/reset', json={'task_id': 'team_phishing_escalation', 'seed': 42, 'mode': 'team'}).json()
print(f"Team mode — alerts: {len(obs['alert_queue'])}, role: {obs.get('current_role')}, phase: {obs.get('current_phase')}")

# Test 3: Red-team generated scenario
gen_resp = client.post('/generate_scenario', json={'seed': 42})
gen_data = gen_resp.json()
print(f"Generated scenario — task_id: {gen_data.get('task_id')}, difficulty: {gen_data.get('difficulty_floor')}")

print('\n✓ Environment verified')

Solo mode — alerts: 1, mode: tier1_solo
Team mode — alerts: 1, role: tier1, phase: triage
Generated scenario — task_id: red_team_generated, difficulty: 0.5

✓ Environment verified


## 4. Baseline: Oracle Heuristic Reward Curves

Run the scripted oracle agent to establish baseline scores before training.

In [3]:
import sys
sys.path.insert(0, '.')
from train_grpo import run_episode, oracle_action, TIER1_TASKS, SEEDS

# Collect baseline scores on 10 seeds × 2 tasks = 20 episodes
baseline_scores = []
print('Running baseline oracle episodes...')
for task_id in TIER1_TASKS:
    for seed in SEEDS[:10]:
        score, traj = run_episode(client, task_id=task_id, seed=seed, role_to_train='tier1')
        score = max(0.0, float(score))  # clamp: negative cumulative = 0 task score
        baseline_scores.append({'task': task_id, 'seed': seed, 'score': score})
        print(f'  {task_id} seed={seed}: {score:.4f}')

baseline_avg = sum(s['score'] for s in baseline_scores) / len(baseline_scores)
print(f'\nOracle baseline avg: {baseline_avg:.4f}')
print('(Scripted heuristic; trained model should exceed this)')

Running baseline oracle episodes...
  team_phishing_escalation seed=42: 0.9990
  team_phishing_escalation seed=43: 0.9990
  team_phishing_escalation seed=44: 0.9990
  team_phishing_escalation seed=45: 0.9990
  team_phishing_escalation seed=46: 0.9990
  team_phishing_escalation seed=47: 0.9990
  team_phishing_escalation seed=48: 0.9990
  team_phishing_escalation seed=49: 0.9990
  team_phishing_escalation seed=50: 0.9990
  team_phishing_escalation seed=51: 0.9990
  team_lateral_team seed=42: 0.8000
  team_lateral_team seed=43: 0.8000
  team_lateral_team seed=44: 0.8000
  team_lateral_team seed=45: 0.8000
  team_lateral_team seed=46: 0.8000
  team_lateral_team seed=47: 0.8000
  team_lateral_team seed=48: 0.8000
  team_lateral_team seed=49: 0.8000
  team_lateral_team seed=50: 0.8000
  team_lateral_team seed=51: 0.8000

Oracle baseline avg: 0.8995
(Scripted heuristic; trained model should exceed this)


## 5. Load Model with Unsloth (4-bit LoRA)

In [4]:
from unsloth import FastLanguageModel
import os

# 3B Instruct on a single T4 (16 GB). Gated — requires HF_TOKEN.
# If 3B OOMs, fall back to Qwen/Qwen2.5-1.5B-Instruct (also gated, 16 GB safe).
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
HF_TOKEN = os.getenv('HF_TOKEN', '')
if not HF_TOKEN:
    print('[warn] HF_TOKEN not set — Instruct models are gated. Set HF_TOKEN or change MODEL_NAME to Qwen/Qwen2.5-0.5B.')

# max_seq_length=896 = 768-token prompt (training/prompts.py, worst case ~680) + 128
# completion. The old 768 fit a 512-token prompt that hid the agent's own progress.
# 3B + group=4 was ~13–14 GB peak at 768; 896 is ~17% more activation memory and
# may OOM on a 15 GB T4. If it does, use Qwen/Qwen2.5-1.5B-Instruct (the size of the
# published adapter) or num_generations=2.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=896,
    dtype=None,          # auto-detect (T4 → fp16)
    load_in_4bit=True,
    token=HF_TOKEN or None,
)

# LoRA r=16: r=32 on a 3B base eats another ~1 GB and risks OOM on T4.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model.print_trainable_parameters()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-04-26 02:47:06.248202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777171626.272299     240 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777171626.280175     240 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777171626.300765     240 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777171626.300786     240 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777171626.300789     240 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
[warn] HF_TOKEN not set — Instruct models are gated. Set HF_TOKEN or change MODEL_NAME to Qwen/Qwen2.5-0.5B.
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## 6. Build Training Dataset from Environment

In [5]:
from datasets import Dataset
from train_grpo import build_step_dataset, ROLE_SYSTEM_PROMPTS

ROLE = 'tier1'  # Change to 'tier2' or 'manager' for staged training
TRAIN_SEEDS = list(range(42, 42 + 40))  # 40 seeds per task — 2x the per-step examples for smoother gradients

# Per-step dataset: one row per (obs, step_index) for the target role.
# This is the core of real per-step GRPO — each completion is scored against
# the immediate env step reward for the exact state it was drawn from.
print(f'Building per-step dataset for role={ROLE}...')
raw_data = build_step_dataset(client, TIER1_TASKS, TRAIN_SEEDS, ROLE)

import random
random.seed(42)
random.shuffle(raw_data)

split = max(1, len(raw_data) // 10)
train_data = raw_data[split:]
eval_data = raw_data[:split]

train_dataset = Dataset.from_list([
    {'prompt': d['prompt'],
     'task_id': d['task_id'],
     'seed': d['seed'],
     'step_index': d.get('step_index', 0)}
    for d in train_data
])
print(f'Train: {len(train_dataset)} | Eval: {len(eval_data)}')


Building per-step dataset for role=tier1...
Train: 900 | Eval: 100


## 7. Define Reward Function

In [6]:
from train_grpo import make_reward_fn

reward_fn = make_reward_fn(client, ROLE)

# Sanity check: per-step reward on a known good T1 action at step 0.
test_completions = [[
    {'role': 'assistant', 'content': json.dumps({
        'action_type': 'enrich_indicator',
        'indicator': '203.57.1.1',
        'indicator_type': 'ip',
        'query_alert_id': 'ALT-TEAM-001',
        'role': 'tier1'
    })}
]]
test_rewards = reward_fn(
    prompts=['test'],
    completions=test_completions,
    task_id=['team_phishing_escalation'],
    seed=[42],
    step_index=[0],
)
print(f'Per-step reward sanity check: {test_rewards}')

Per-step reward sanity check: [0.038000000000000006]


## 8. GRPO Training (group=8)

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import trl, inspect, torch

print(f'TRL version: {trl.__version__}')

# Detect precision support. T4 (Turing) does NOT support bf16 — it needs fp16.
# A100/H100 (Ampere+) support bf16. TRL 0.24 defaults bf16=True, which crashes
# on T4, so we set it explicitly based on the actual GPU capability.
_supports_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"} | bf16={_supports_bf16}')

# Build config kwargs — newer TRL dropped `beta` and changed some defaults.
# We detect what the current version accepts rather than hardcoding.
_grpo_params = set(inspect.signature(GRPOConfig.__init__).parameters)

# Memory budget for 3B on T4 (16 GB):
#   group=4 (was 16) — single biggest VRAM lever in GRPO
#   micro-batch=1 (was 2)
#   grad_accum=16 → effective batch still 16
#   prompt 512 (was 768) — shorter rollouts = less KV cache
config_kwargs = dict(
    output_dir=f'./soc_grpo_{ROLE}',
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=4,                  # T4-safe; group=4 still gives usable advantages
    max_prompt_length=768,
    max_completion_length=128,
    learning_rate=3e-6,
    temperature=1.0,
    logging_steps=2,
    save_steps=50,
    report_to=['none'],
    seed=42,
    bf16=_supports_bf16,
    fp16=not _supports_bf16 and torch.cuda.is_available(),
)

# beta= (KL penalty) exists in TRL ≤ 0.11; newer versions removed it
if 'beta' in _grpo_params:
    config_kwargs['beta'] = 0.04
elif 'kl_coef' in _grpo_params:
    config_kwargs['kl_coef'] = 0.04

grpo_config = GRPOConfig(**config_kwargs)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print(f'Training {len(train_dataset)} examples × 5 epochs, GRPO group=4 (T4-safe)')
trainer.train()


TRL version: 0.24.0
GPU: Tesla T4 | bf16=False
Training 900 examples × 5 epochs, GRPO group=4 (T4-safe)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 900 | Num Epochs = 2 | Total steps = 900
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


[reward_fn] parse: strict 100% loose 0% fallback 0% | mean reward +0.050


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_fn / mean,rewards / reward_fn / std
2,0.000000,0.047500,0.005000,48.750000,24.000000,101.500000,0.062500,44.151787,24.000000,81.500000,0.000033,0.047500,0.007071
4,0.000000,0.050000,0.000000,47.437500,17.500000,82.500000,0.000000,47.437500,17.500000,82.500000,0.000049,0.050000,0.000000
6,0.000000,0.050000,0.000000,32.437500,17.500000,69.500000,0.000000,32.437500,17.500000,69.500000,0.000025,0.050000,0.000000
8,0.000000,0.050000,0.000000,37.750000,17.500000,81.500000,0.000000,37.750000,17.500000,81.500000,0.000013,0.050000,0.000000
10,0.000000,0.050000,0.000000,33.625000,15.000000,70.000000,0.000000,33.625000,15.000000,70.000000,0.000063,0.050000,0.000000
12,0.000000,0.040625,0.018750,54.062500,17.500000,108.500000,0.062500,49.419643,17.500000,100.000000,0.000025,0.040625,0.026517
14,0.000000,0.050000,0.000000,37.875000,15.000000,68.500000,0.000000,37.875000,15.000000,68.500000,0.000065,0.050000,0.000000
16,0.000000,0.050000,0.000000,41.250000,17.500000,61.000000,0.000000,41.250000,17.500000,61.000000,0.000034,0.050000,0.000000
18,0.000000,0.040625,0.018750,35.875000,15.000000,78.000000,0.000000,35.875000,15.000000,78.000000,0.000036,0.040625,0.026517
20,0.000000,0.050000,0.000000,33.625000,15.000000,71.000000,0.000000,33.625000,15.000000,71.000000,0.000017,0.050000,0.000000


[reward_fn] parse: strict 88% loose 12% fallback 0% | mean reward +0.045
[reward_fn] parse: strict 100% loose 0% fallback 0% | mean reward +0.050
[reward_fn] parse: strict 100% loose 0% fallback 0% | mean reward +0.050
[reward_fn] parse: strict 100% loose 0% fallback 0% | mean reward +0.050
[reward_fn] parse: strict 100% loose 0% fallback 0% | mean reward +0.050
[reward_fn] parse: strict 100% loose 0% fallback 0% | mean reward +0.050
[reward_fn] parse: strict 100% loose 0% fallback 0% | mean reward +0.050
[reward_fn] parse: strict 100% loose 0% fallback 0% | mean reward +0.050
[reward_fn] parse: strict 100% loose 0% fallback 0% | mean reward +0.050
[reward_fn] parse: strict 100% loose 0% fallback 0% | mean reward +0.050
[reward_fn] parse: strict 88% loose 0% fallback 12% | mean reward +0.031
[reward_fn] parse: strict 100% loose 0% fallback 0% | mean reward +0.050
[reward_fn] parse: strict 100% loose 0% fallback 0% | mean reward +0.050
[reward_fn] parse: strict 100% loose 0% fallback 0%

## 9. Save Model

In [7]:
output_path = f'./soc_grpo_{ROLE}'
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)
print(f'Model saved to {output_path}')

# Optional: push to HuggingFace Hub
# model.push_to_hub(f'YOUR_HF_USERNAME/soc-triage-gym-{ROLE}', token=HF_TOKEN)

NameError: name 'trainer' is not defined

## 10. Post-Training Evaluation — Reward Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from unsloth import FastLanguageModel
from train_grpo import parse_action_from_text, format_obs_prompt

FastLanguageModel.for_inference(model)

def run_trained_episode(task_id, seed):
    """Run a team episode where the trained model handles the target role."""
    obs = client.post('/reset', json={'task_id': task_id, 'seed': seed, 'mode': 'team'}).json()
    step = 0
    while not obs.get('done', False) and step < 80:
        step += 1
        role = obs.get('current_role', 'tier1')
        if role == ROLE:
            prompt_text = format_obs_prompt(obs, role, step)
            messages = [
                {'role': 'system', 'content': ROLE_SYSTEM_PROMPTS[ROLE]},
                {'role': 'user', 'content': prompt_text}
            ]
            inputs = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
            ).to(model.device)
            outputs = model.generate(
                inputs, max_new_tokens=128, temperature=0.0, do_sample=False
            )
            text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
            action = parse_action_from_text(text, role)
        else:
            action = oracle_action(obs)
        step_resp = client.post('/step', content=json.dumps(action),
                                headers={'Content-Type': 'application/json'})
        obs = step_resp.json()
    return obs.get('task_score') or obs.get('cumulative_reward', 0.0)

# Evaluate on held-out seeds — 30 seeds × 2 tasks = 60 points for a smoother curve.
EVAL_SEEDS = list(range(100, 130))
trained_scores = []
print('Evaluating trained model...')
for task_id in TIER1_TASKS:
    for seed in EVAL_SEEDS:
        score = run_trained_episode(task_id, seed)
        trained_scores.append(score)
        print(f'  {task_id} seed={seed}: {score:.4f}')

trained_avg = sum(trained_scores) / len(trained_scores)
print(f'\nTrained model avg: {trained_avg:.4f}')
print(f'Oracle baseline avg: {baseline_avg:.4f}')
print(f'Improvement: {trained_avg - baseline_avg:+.4f} ({(trained_avg/baseline_avg - 1)*100:+.1f}%)')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('SOC-Triage-Gym v2 — GRPO Training Results', fontsize=14, fontweight='bold')

# Panel 1: before/after bar
ax = axes[0]
ax.bar(['Oracle Baseline', f'GRPO Trained\n(role={ROLE})'],
       [baseline_avg, trained_avg],
       color=['#4C72B0', '#DD8452'], alpha=0.85, edgecolor='black')
ax.set_ylim(0, max(1.0, trained_avg * 1.2))
ax.set_ylabel('Episode Score')
ax.set_title('Before vs After Training')
ax.axhline(0.5, linestyle='--', color='gray', alpha=0.5, label='0.5 threshold')
for i, v in enumerate([baseline_avg, trained_avg]):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# Panel 2: per-episode held-out evaluation curve
ax2 = axes[1]
bl_vals = [max(0.0, s['score']) for s in baseline_scores[:len(trained_scores)]]
x = range(1, len(trained_scores) + 1)
ax2.plot(x, bl_vals, 'o-', alpha=0.5, label='Oracle baseline', color='#4C72B0')
ax2.plot(x, trained_scores, 's-', alpha=0.7, label=f'GRPO trained ({ROLE})', color='#DD8452')
w = min(11, len(trained_scores))                  # 5 → 11: stronger smoothing for the larger 60-pt series
if len(trained_scores) >= w:
    sm = np.convolve(trained_scores, np.ones(w)/w, mode='valid')
    ax2.plot(range(w, len(trained_scores)+1), sm, linewidth=2.5, color='#C44E52', label='Trained (smoothed)')
ax2.set_xlabel('Evaluation Episode'); ax2.set_ylabel('Score')
ax2.set_title('Per-Episode Scores (Held-Out Seeds)')
ax2.set_ylim(0, max(1.0, max(trained_scores) * 1.1) if trained_scores else 1.0)
ax2.legend()

# Panel 3: TRL training reward curve from trainer.state.log_history.
# logging_steps=2 gives a dense series; show raw + smoothed.
ax3 = axes[2]
log = getattr(trainer.state, 'log_history', []) or []
reward_pts = [(rec.get('step', i), rec.get('reward', rec.get('train/reward')))
              for i, rec in enumerate(log)
              if ('reward' in rec) or ('train/reward' in rec)]
reward_pts = [(s, r) for s, r in reward_pts if r is not None]
if reward_pts:
    steps, rewards = zip(*reward_pts)
    ax3.plot(steps, rewards, 'o-', alpha=0.35, color='#55A868', label='Train reward')
    rw = min(15, max(3, len(rewards)//5))
    if len(rewards) >= rw:
        sm = np.convolve(rewards, np.ones(rw)/rw, mode='valid')
        ax3.plot(steps[rw-1:], sm, linewidth=2.5, color='#1F6B3A', label=f'Smoothed (w={rw})')
    ax3.set_xlabel('Training Step'); ax3.set_ylabel('Mean Group Reward')
    ax3.set_title('GRPO Training Reward')
    ax3.legend()
else:
    ax3.text(0.5, 0.5, 'No reward in trainer.state.log_history\n(check logging_steps)',
             ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('GRPO Training Reward')

plt.tight_layout()
plt.savefig('soc_grpo_results.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: soc_grpo_results.png')

# Save to Google Drive if mounted
import os
drive_path = '/content/drive/MyDrive/soc_triage_gym_results'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_path, exist_ok=True)
    plt.savefig(f'{drive_path}/soc_grpo_results.png', dpi=200, bbox_inches='tight')
    print(f'Also saved to Google Drive: {drive_path}/soc_grpo_results.png')


## 11. Red-Team Curriculum — Blue Win Rate Oscillation

This is the **Theme #4 Self-Improvement money shot**: blue-team win rate oscillates around 0.5 as the Red-Team Generator adapts difficulty.

In [ ]:
from scenarios.red_team_generator import RedTeamGenerator
from models import RedTeamConfig
import matplotlib.pyplot as plt
import numpy as np

# Heuristic curriculum (no model training) — cranking these is essentially free.
NUM_ROUNDS = 40            # 15 → 40: more rounds = clearer oscillation
EPISODES_PER_ROUND = 15    # 5 → 15: better win-rate denominator (no more {0, 0.2, 0.4} buckets)

generator = RedTeamGenerator(seed=42)
difficulty_history = []
blue_win_rate_history = []

print('Running Red-Team curriculum simulation...')
for round_num in range(NUM_ROUNDS):
    round_scores = []
    for seed_offset in range(EPISODES_PER_ROUND):
        # Generate scenario
        gen_resp = client.post('/generate_scenario', json={
            'difficulty_floor': generator.config.difficulty_floor,
            'noise_density': generator.config.noise_density,
            'seed': generator.seed + seed_offset
        })
        if gen_resp.status_code != 200:
            round_scores.append(0.5)
            continue
        # Blue team plays (oracle heuristic)
        obs = client.post('/reset', json={'task_id': 'red_team_generated', 'seed': generator.seed + seed_offset, 'mode': 'team'}).json()
        step = 0
        while not obs.get('done', False) and step < 50:
            step += 1
            action = oracle_action(obs)
            obs = client.post('/step', content=json.dumps(action),
                             headers={'Content-Type': 'application/json'}).json()
        score = obs.get('task_score') or obs.get('cumulative_reward', 0.0)
        round_scores.append(float(score))

    blue_win_rate = sum(1 for s in round_scores if s > 0.5) / len(round_scores)
    diff = generator.config.difficulty_floor

    difficulty_history.append(diff)
    blue_win_rate_history.append(blue_win_rate)

    print(f'  Round {round_num+1:2d} | difficulty={diff:.2f} | blue_win_rate={blue_win_rate:.2f}')

    # Adapt difficulty for next round
    generator = generator.adapt_difficulty(blue_win_rate)

# Plot the oscillation
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('SOC-Triage-Gym v2 — Red-Team Curriculum (Theme #4 Self-Improvement)', fontweight='bold')

rounds = np.arange(1, NUM_ROUNDS + 1)
ax1.plot(rounds, blue_win_rate_history, 'o-', color='#2196F3', linewidth=1.5,
         alpha=0.45, label='Blue win rate (raw)')
# Smoothed line for the win-rate trend.
sw = min(5, NUM_ROUNDS)
if NUM_ROUNDS >= sw:
    sm = np.convolve(blue_win_rate_history, np.ones(sw)/sw, mode='valid')
    ax1.plot(rounds[sw-1:], sm, color='#0D47A1', linewidth=2.5, label=f'Smoothed (w={sw})')
ax1.axhline(0.5, linestyle='--', color='red', alpha=0.7, label='Target 0.5')
ax1.fill_between(rounds, 0.4, 0.6, alpha=0.1, color='green', label='Sweet spot [0.4, 0.6]')
ax1.set_ylabel('Blue Win Rate')
ax1.set_ylim(0, 1)
ax1.legend()
ax1.set_title('Blue-Team Win Rate (oscillates around 0.5 as Red-Team adapts)')

ax2.plot(rounds, difficulty_history, 's-', color='#F44336', linewidth=2, label='Difficulty floor')
ax2.set_xlabel('Round')
ax2.set_ylabel('Difficulty Floor')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.set_title('Red-Team Difficulty Adaptation (escalates when blue wins too often)')

plt.tight_layout()
plt.savefig('redteam_curriculum.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: redteam_curriculum.png')


## 12. Summary

### Results

| Metric | Value |
|--------|-------|
| Oracle baseline avg score | See `baseline_avg` printed above |
| GRPO trained model avg score | See `trained_avg` printed above |
| Improvement | `trained_avg - baseline_avg` |
| Tasks trained on | team_phishing_escalation, team_lateral_team |
| Training role | Tier-1 (frozen T2 + Manager as oracle) |
| GRPO group size | 8 |
| Training examples | ~800 per-step rows (20 seeds × 2 tasks × ~20 T1 steps each) |
| Red-Team curriculum rounds | 15 |

### What makes this different from other OpenEnv submissions

- **Per-step GRPO reward**: each training example is one `(obs, step_index)` pair. Reward = env's immediate `step_reward` after applying the model's action at that state — not a full-episode oracle score.
- **Team reward blend**: `0.6 × role + 0.4 × Δteam_F1` — agents are penalized for solo-optimizing at the expense of team outcome.
- **Fleet AI oversight**: SOC Manager role uses an LLM judge (heuristic fallback) to score natural-language explanations of the team's decisions.
- **Self-improving curriculum**: Red-Team Generator adapts difficulty round-by-round, tracking blue-team skill level rather than using a fixed schedule.
- **Reward integrity**: 6 exploit vectors audited and fixed; all regression-tested.

### Theme coverage
- **Theme #1 Multi-Agent Interactions**: T1 → T2 → Manager with ticket bus and phase state machine
- **Fleet AI Scalable Oversight**: Manager audits, flags, and explains team behavior — scored by LLM judge
- **Theme #4 Self-Improvement**: Red-Team Generator co-evolves; curriculum plot shows oscillation

### Output files
- `soc_grpo_results.png` — before/after GRPO reward comparison
- `redteam_curriculum.png` — Theme #4 self-improvement oscillation curve
- `./soc_grpo_tier1/` — trained Tier-1 LoRA adapter weights